## Revised Stage 1 structure

                Stage 1 — Data Ingestion
                            │
                            ├── 1.1 LangChain Document Abstraction          ✅
                            │
                            ├── 1.2 TXT Ingestion                            ✅
                            │
                            ├── 1.3 TextLoader Deep Dive                    ✅
                            │
                            ├── 1.4 PDF Ingestion                            ✅
                            │
                            ├── 1.4.1 PDF Loader Comparison                  🔄
                            │       ├── PyPDFLoader
                            │       └── PyMuPDFLoader
                            │
                            ├── 1.4.2 OCR & Complex PDF Ingestion            ← NEXT
                            │       ├── Text-based PDF
                            │       ├── Scanned PDF
                            │       ├── OCR
                            │       ├── Tables
                            │       ├── Images
                            │       ├── Multi-column layouts
                            │       ├── Headers / footers
                            │       ├── Extraction problems
                            │       └── RAG ingestion strategy
                            │
                            └── 1.5 DOCX Ingestion

## Stage 1.4.2 — OCR & Complex PDF Ingestion

We'll structure this as a set of experiments in the same notebook-first style you've been following.

                            Stage 1.4.2
                            │
                            ├── 1.4.2.1 Understand PDF types
                            │       ├── Text-based PDF
                            │       └── Scanned/Image-based PDF
                            │
                            ├── 1.4.2.2 Detect whether a PDF has extractable text
                            │
                            ├── 1.4.2.3 Experiment with a scanned PDF
                            │
                            ├── 1.4.2.4 Why normal PDF loaders fail
                            │
                            ├── 1.4.2.5 OCR
                            │       └── Image → Text
                            │
                            ├── 1.4.2.6 OCR → LangChain Document
                            │
                            ├── 1.4.2.7 Complex PDF layouts
                            │       ├── Tables
                            │       ├── Multiple columns
                            │       ├── Headers / footers
                            │       ├── Images
                            │       └── Mixed content
                            │
                            ├── 1.4.2.8 Extraction quality
                            │
                            └── 1.4.2.9 RAG ingestion strategy

## The key architecture we'll build

                               PDF
                                │
                                ▼
                        ┌───────────────┐
                        │ PDF Analysis  │
                        └───────┬───────┘
                                │
                        ┌───────┴────────┐
                        │                │
                    Text exists?       No text
                        │                │
                        ▼                ▼
                Text Extraction        OCR
                        │                │
                        └───────┬────────┘
                                ▼
                        Content Extraction
                                │
                                ▼
                        Normalization
                                │
                                ▼
                        LangChain Documents
                                │
                                ▼
                            Chunking
                            ↓
                        Embedding
                            ↓
                            Indexing
                            ↓
                        Retrieval

### Complex PDFs

We'll also investigate a PDF like:


┌─────────────────────────────────────┐
│ Header                              │
├───────────────┬─────────────────────┤
│ Column 1       │ Column 2           │
│                │                    │
│ Text           │ Table              │
│ Text           │ ┌───┬───┬───┐      │
│                │ │ A │ B │ C │      │
│                │ └───┴───┴───┘      │
│                │                    │
│                │ Chart/Image        │
├────────────────┴────────────────────┤
│ Footer                              │
└─────────────────────────────────────┘


### What we'll use

For the initial experiments, we'll stay with the Python/LangChain ecosystem you're already using and introduce OCR only where necessary.

We'll likely encounter tools/libraries such as:

PyMuPDF
pypdf
OCR tooling
LangChain document abstractions
potentially specialized PDF/document parsers

But we'll introduce each only when the experiment requires it, rather than installing a huge collection of packages upfront.

## Stage 1.4.2.1 — Text-based PDF vs Scanned PDF

### The important difference is:

                    Normal PDF
                        ↓
                    Contains actual text
                        ↓
                    PDF text extractor can read it

                    Scanned PDF
                        ↓
                    Contains an image of text
                        ↓
                    PDF text extractor cannot directly read the text
                        ↓
                    OCR is required


## Objective

Understand the fundamental difference between a text-based PDF and a scanned
PDF by loading both with the same PDF extraction approach and observing the
actual extracted content.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

In [ ]:
from pathlib import Path

text_pdf_path = Path(
    "../../data/raw/pdf/azure_event_hubs_knowledge.pdf"
)

scanned_pdf_path = Path(
    "../../data/raw/pdf/azure_event_hubs_scanned.pdf"
)

print("Text PDF:")
print(text_pdf_path.resolve())
print("Exists:", text_pdf_path.exists())

print()

print("Scanned PDF:")
print(scanned_pdf_path.resolve())
print("Exists:", scanned_pdf_path.exists())

In [ ]:
text_loader = PyMuPDFLoader(str(text_pdf_path))

In [ ]:
text_documents = text_loader.load()

In [ ]:
print("Number of Documents:", len(text_documents))

In [ ]:
print(text_documents[0].page_content)

In [ ]:
print(text_documents[0].metadata)

### Now load the scanned PDF

In [ ]:
scanned_loader = PyMuPDFLoader(str(scanned_pdf_path))

In [ ]:
scanned_documents = scanned_loader.load()

In [ ]:
scanned_documents = scanned_loader.load()

#### So far, everything looks normal.

This is intentional.

#### Now inspect the content

In [ ]:
print(scanned_documents[0].page_content)

### effectively no useful content.

## Observation

The scanned PDF visibly contains text when opened by a human.

However, PyMuPDFLoader cannot extract the visible text because the page
contains an image rather than an actual PDF text layer.

Therefore:

Human → can see the text

PDF text extractor → cannot directly read the text

This is where OCR becomes necessary.

### Let's prove it using character count

In [ ]:
print(
    "Text-based PDF characters:",
    len(text_documents[0].page_content)
)

print(
    "Scanned PDF characters:",
    len(scanned_documents[0].page_content)
)

## Compare the two Documents

In [ ]:
### Compare the two Documents
# text-based PDF comparision

print("TEXT-BASED PDF")
print("=" * 60)

print("Content:")
print(text_documents[0].page_content[:500])

print("\nMetadata:")
print(text_documents[0].metadata)

print("\nCharacter count:")
print(len(text_documents[0].page_content))

In [ ]:
print("\nSCANNED PDF")
print("=" * 60)

print("Content:")
print(scanned_documents[0].page_content[:500])

print("\nMetadata:")
print(scanned_documents[0].metadata)

print("\nCharacter count:")
print(len(scanned_documents[0].page_content))

## Let's create a simple detection experiment

In [ ]:
def inspect_pdf_documents(name, documents):

    print("=" * 70)
    print(name)
    print("=" * 70)

    print("Number of Documents:", len(documents))

    for index, document in enumerate(documents):

        content = document.page_content.strip()

        print(f"\nDocument {index + 1}")
        print("Characters:", len(content))
        print("Has text:", bool(content))
        print("Metadata:", document.metadata)

In [ ]:
inspect_pdf_documents("Text-based PDF",text_documents)

In [ ]:
inspect_pdf_documents("Scanned PDF",scanned_documents)

## 1.4.2.1 Key Takeaways

1. Not all PDFs contain an actual text layer.
2. A text-based PDF contains machine-readable text.
3. A scanned PDF may contain only images of text.
4. Humans can visually read a scanned PDF even when a text extractor cannot.
5. PyMuPDFLoader can extract text from text-based PDFs.
6. A normal PDF text extractor cannot automatically perform OCR on an image-only page.
7. OCR converts text contained in images into machine-readable text.
8. OCR becomes an important part of RAG ingestion for scanned documents.
9. Having some extracted text does not necessarily mean extraction quality is good.
10. Poor document extraction propagates problems into chunking, embeddings, indexing, and retrieval.


                    PDF
                     │
                     ▼
               Inspect Pages
                     │
           ┌─────────┴─────────┐
           ▼                   ▼
     Text available        No text
           │                   │
           ▼                   ▼
      Normal extraction       OCR

